# Dataset intake and structural validation

BPI Challenge 2015 Municipality 1. This notebook reads the local XES file without modifying it. No cleaning, target definition, feature engineering, or modeling is performed.

# 1. Imports

Import only the tools needed for intake.

In [1]:
from pathlib import Path
import pandas as pd
import pm4py



  Welcome to PM4Py — Community Version
  Open-Source License (AGPL v3)

  📚 Docs & Examples:
     https://processintelligence.solutions/pm4py

  ⚖️  License: AGPL v3 — Commercial use requires open-sourcing your application.
     Business use without open-sourcing? A commercial license is available:
     https://processintelligence.solutions/pm4py#licensing




# 2. Dataset path

Search the current directory and its parents for the project root. This works from the project root, notebooks folder, or another project subfolder; start Jupyter inside this project.

In [2]:
candidates = [Path.cwd(), *Path.cwd().parents]
project_root = next(
    (folder for folder in candidates
     if (folder / "notebooks" / "01_data_loading.ipynb").is_file()
     and (folder / "src").is_dir()),
    None,
)
if project_root is None:
    raise FileNotFoundError("Start Jupyter from inside the capstone project.")
xes_path = project_root / "data" / "raw" / "BPIC15_1.xes"
if not xes_path.is_file():
    raise FileNotFoundError("Place BPIC15_1.xes in data/raw/ before running this notebook.")
print("Dataset:", xes_path.relative_to(project_root))

Dataset: data\raw\BPIC15_1.xes


# 3. Load XES

Read the XES as a PM4Py event log. This operation does not write to the raw file.

In [3]:
event_log = pm4py.read_xes(str(xes_path), return_legacy_log_object=True)
print("Event log loaded successfully.")

C:\SPAR_CAPSTONE\.venv\Lib\site-packages\pm4py\utils.py:1027: UserWarning: Install the optional requirement `r4pm` to import/export files faster. `rustxes` remains supported as a fallback.
  warnings.warn(


Event log loaded successfully.


# 4. Convert to DataFrame

Create an in-memory table of event rows.

In [4]:
df = pm4py.convert_to_dataframe(event_log)
if not isinstance(df, pd.DataFrame):
    raise TypeError("Expected a pandas DataFrame.")
print("Converted event log to pandas DataFrame.")

Converted event log to pandas DataFrame.


# 5. Basic dataset dimensions

Memory usage is approximate and includes the index.

In [5]:
event_rows = len(df)
memory_bytes = int(df.memory_usage(index=True, deep=True).sum())
print("DataFrame shape:", df.shape)
print("Event rows:", event_rows)
print(f"Approximate memory: {memory_bytes:,} bytes ({memory_bytes / 1024**2:.2f} MiB)")

DataFrame shape: (52217, 29)
Event rows: 52217
Approximate memory: 67,422,202 bytes (64.30 MiB)


# 6. Columns

List every column, without truncation.

In [6]:
for position, column in enumerate(df.columns, start=1):
    print(f"{position}. {column}")

1. question
2. dateFinished
3. dueDate
4. action_code
5. activityNameEN
6. planned
7. time:timestamp
8. monitoringResource
9. org:resource
10. activityNameNL
11. concept:name
12. lifecycle:transition
13. case:endDate
14. case:caseStatus
15. case:SUMleges
16. case:last_phase
17. case:case_type
18. case:concept:name
19. case:Responsible_actor
20. case:parts
21. case:termName
22. case:endDatePlanned
23. case:startDate
24. case:requestComplete
25. case:IDofConceptCase
26. case:landRegisterID
27. case:caseProcedure
28. case:Includes_subCases
29. dateStop


# 7. Data types

Inspect the dtype of every column.

In [7]:
print(df.dtypes.to_string())

question                                  str
dateFinished                              str
dueDate                   datetime64[us, UTC]
action_code                               str
activityNameEN                            str
planned                   datetime64[us, UTC]
time:timestamp            datetime64[us, UTC]
monitoringResource                        str
org:resource                              str
activityNameNL                            str
concept:name                              str
lifecycle:transition                      str
case:endDate              datetime64[us, UTC]
case:caseStatus                           str
case:SUMleges                         float64
case:last_phase                           str
case:case_type                            str
case:concept:name                         str
case:Responsible_actor                    str
case:parts                                str
case:termName                             str
case:endDatePlanned       datetime

# 8. Missing values

Count missing values without filling or dropping anything.

In [8]:
missing_values = pd.DataFrame({
    "column": df.columns,
    "missing_count": df.isna().sum().to_numpy(),
    "missing_percent": (df.isna().mean() * 100).to_numpy(),
}).sort_values("missing_percent", ascending=False, kind="stable").reset_index(drop=True)
print(missing_values.to_string(index=False))

                column  missing_count  missing_percent
              dateStop          52215        99.996170
               dueDate          48877        93.603616
    case:caseProcedure          46538        89.124232
   case:endDatePlanned          45090        86.351188
   case:landRegisterID          44210        84.665913
  case:IDofConceptCase          16381        31.371009
         case:termName          13743        26.319015
case:Includes_subCases          11280        21.602160
         case:SUMleges           8632        16.531015
          case:endDate           7744        14.830419
               planned           6013        11.515407
            case:parts             23         0.044047
case:Responsible_actor             19         0.036387
           action_code              1         0.001915
              question              0         0.000000
          dateFinished              0         0.000000
        activityNameEN              0         0.000000
        ti

# 9. First rows

Display the first five event rows, including all columns.

In [9]:
with pd.option_context("display.max_columns", None):
    display(df.head(5))

,question,dateFinished,dueDate,action_code,activityNameEN,planned,time:timestamp,monitoringResource,org:resource,activityNameNL,concept:name,lifecycle:transition,case:endDate,case:caseStatus,case:SUMleges,case:last_phase,case:case_type,case:concept:name,case:Responsible_actor,case:parts,case:termName,case:endDatePlanned,case:startDate,case:requestComplete,case:IDofConceptCase,case:landRegisterID,case:caseProcedure,case:Includes_subCases,dateStop
0,EMPTY,2014-04-14 00:00:00,2014-04-17 07:30:39+00:00,01_HOOFD_010,register submission date request,2014-04-15 07:30:39+00:00,2014-04-10 22:00:00+00:00,2670601,9264148,registratie datum binnenkomst aanvraag,01_HOOFD_010,complete,2014-07-17 22:00:00+00:00,O,157.09305,Beschikking verzonden,557669,10009138,560464,"Bouw,Reclame",Termijn bezwaar en beroep 1,2014-07-17 22:00:00+00:00,2014-04-10 22:00:00+00:00,TRUE,10009280,NaN,NaN,NaN,NaN
1,False,2014-04-14 00:00:00,NaT,01_HOOFD_011,OLO messaging active,2014-04-15 07:34:03+00:00,2014-04-13 22:00:00+00:00,2670601,9264148,OLO berichtenverkeer actief,01_HOOFD_011,complete,2014-07-17 22:00:00+00:00,O,157.09305,Beschikking verzonden,557669,10009138,560464,"Bouw,Reclame",Termijn bezwaar en beroep 1,2014-07-17 22:00:00+00:00,2014-04-10 22:00:00+00:00,TRUE,10009280,NaN,NaN,NaN,NaN
2,True,2014-04-14 00:00:00,NaT,01_HOOFD_020,send confirmation receipt,2014-04-15 07:34:03+00:00,2014-04-13 22:00:00+00:00,2670601,9264148,versturen ontvangstbevestiging,01_HOOFD_020,complete,2014-07-17 22:00:00+00:00,O,157.09305,Beschikking verzonden,557669,10009138,560464,"Bouw,Reclame",Termijn bezwaar en beroep 1,2014-07-17 22:00:00+00:00,2014-04-10 22:00:00+00:00,TRUE,10009280,NaN,NaN,NaN,NaN
3,False,2014-04-14 00:00:00,2014-04-21 07:37:17+00:00,02_DRZ_010,forward to the competent authority,2014-04-15 07:37:17+00:00,2014-04-13 22:00:00+00:00,2670601,9264148,doorsturen aan bevoegd gezag,02_DRZ_010,complete,2014-07-17 22:00:00+00:00,O,157.09305,Beschikking verzonden,557669,10009138,560464,"Bouw,Reclame",Termijn bezwaar en beroep 1,2014-07-17 22:00:00+00:00,2014-04-10 22:00:00+00:00,TRUE,10009280,NaN,NaN,NaN,NaN
4,True,2014-04-14 00:00:00,NaT,04_BPT_005,regular procedure without MER,2014-04-15 07:37:22+00:00,2014-04-13 22:00:00+00:00,2670601,9264148,reguliere procedure zonder MER,04_BPT_005,complete,2014-07-17 22:00:00+00:00,O,157.09305,Beschikking verzonden,557669,10009138,560464,"Bouw,Reclame",Termijn bezwaar en beroep 1,2014-07-17 22:00:00+00:00,2014-04-10 22:00:00+00:00,TRUE,10009280,NaN,NaN,NaN,NaN


# 10. Process-log sanity checks

Identify standard XES columns and inspect structural counts. Activity counts are reported separately for each available code/name column. Missing resources are excluded from unique counts.

In [10]:
case_column = "case:concept:name"
activity_columns = [name for name in ["concept:name", "action_code", "activityNameEN", "activityNameNL"] if name in df.columns]
timestamp_column = "time:timestamp"
resource_column = "org:resource" if "org:resource" in df.columns else None
required = [case_column, "concept:name", timestamp_column]
missing_required = [name for name in required if name not in df.columns]
if missing_required:
    raise ValueError(f"Missing required process columns: {missing_required}")
case_count = int(df[case_column].nunique())
activity_counts = {name: int(df[name].nunique()) for name in activity_columns}
resource_count = int(df[resource_column].nunique()) if resource_column else None
events_per_case = df.groupby(case_column, dropna=False).size()
events_per_case_stats = {
    "mean": float(events_per_case.mean()),
    "median": float(events_per_case.median()),
    "min": int(events_per_case.min()),
    "max": int(events_per_case.max()),
}
minimum_timestamp = df[timestamp_column].min()
maximum_timestamp = df[timestamp_column].max()
print("Case ID:", case_column)
print("Activity code/name columns:", activity_columns)
print("Timestamp:", timestamp_column)
print("Resource:", resource_column)
print("Unique cases:", case_count)
print("Unique activities by column:", activity_counts)
print("Unique resources:", resource_count)
print("Earliest timestamp:", minimum_timestamp)
print("Latest timestamp:", maximum_timestamp)
print("Events per case:", events_per_case_stats)
print("Missing key values:")
print(df[[*required, *([resource_column] if resource_column else [])]].isna().sum().to_string())

Case ID: case:concept:name
Activity code/name columns: ['concept:name', 'action_code', 'activityNameEN', 'activityNameNL']
Timestamp: time:timestamp
Resource: org:resource
Unique cases: 1199
Unique activities by column: {'concept:name': 398, 'action_code': 397, 'activityNameEN': 289, 'activityNameNL': 289}
Unique resources: 23
Earliest timestamp: 2010-10-04 22:00:00+00:00
Latest timestamp: 2015-07-31 22:00:00+00:00
Events per case: {'mean': 43.55045871559633, 'median': 44.0, 'min': 2, 'max': 101}
Missing key values:
case:concept:name    0
concept:name         0
time:timestamp       0
org:resource         0


# 11. Duplicate event-row check

Fully duplicated rows are reported without removal. Apparent duplicate rows in an event log may require domain interpretation before removal; repeated events can be meaningful.

In [11]:
duplicate_rows = int(df.duplicated().sum())
print("Fully duplicated event rows:", duplicate_rows)

Fully duplicated event rows: 0


# 12. Validation summary

- Dataset loaded successfully from the local XES file.
- Event rows: 52,217; columns: 29.
- Cases: 1,199.
- Time range: 2010-10-04 22:00:00+00:00 to 2015-07-31 22:00:00+00:00.
- Case identifier: `case:concept:name`.
- Activity code/name columns: `concept:name`, `action_code`, `activityNameEN`, `activityNameNL`.
- Timestamp: `time:timestamp` (datetime64[us, UTC]).
- Resource: `org:resource`.
- Structural observations: Missing values occur in 14 columns (see Section 8); no imputation performed.
- No rows were modified, dropped, imputed, or deduplicated. The source XES was read only.
- SHA-256 checks before and after execution match for both local raw files.

This is structural validation only. No full EDA, target definition, feature engineering, processed dataset, or model was created.
